## Gold IncrementalWork

### Step 1: Import and Setup

In [0]:
from pyspark.sql import functions as F
from datetime import datetime
from delta.tables import DeltaTable
import uuid

In [0]:
gold_run_id = str(uuid.uuid4())
run_ts_str = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
run_date_str = datetime.utcnow().strftime("%Y-%m-%d")

### Step 2: Helper Functions

In [0]:
def last_successful_processed_ts(table_id: int):
    control_df = (
        spark.read.table("gold.control.processing_control")
        .filter(
            (F.col("table_id") == F.lit(table_id))
            & (F.col("status") == F.lit("success"))
        )
        .orderBy(F.col("updated_at").desc())
        .limit(1)
    )

    rows = control_df.collect()
    if not rows:
        return None
    return rows[0]["last_processed_at"]

In [0]:
def upsert_to_gold(source_df, target_table, join_key):
    if spark.catalog.tableExists(target_table):
        target_tbl = DeltaTable.forName(spark, target_table)
        (
            target_tbl.alias("t")
            .merge(source_df.alias("s"), f"t.{target_table} = s.{join_key}")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        source_df.write.format("delta").saveAsTable(target_table)

In [0]:
def upsert_gold_control(table_id, last_processed_at, rows_merged, status):
    source_df = spark.createDataFrame(
        [
            (
                gold_run_id,
                table_id,
                last_processed_at,
                rows_merged,
                status,
                datetime.utcnow(),
            )
        ],
        schema="""
            run_id string,
            table_id int, 
            last_processed_at timestamp, 
            rows_merged bigint, 
            status string, 
            updated_at timestamp
        """,
    )

    target_tbl = DeltaTable.forName(spark, "gold.control.processing_control")

    (
        target_tbl.alias("t")
        .merge(source_df.alias("s"), "t.table_id = s.table_id and t.run_id = s.run_id")
        .whenMatchedUpdate(
            set={
                "t.last_processed_at": "s.last_processed_at",
                "t.rows_merged": "s.rows_merged",
                "t.status": "s.status",
                "t.updated_at": "s.updated_at",
            }
        )
    ).whenNotMatchedInsertAll().execute()

### Step 3: Gold Incremental Load

In [0]:
object_list_df = spark.read.table("gold.control.object_list").filter(
    F.col("layer ") == F.lit("gold")
)
object_list = object_list_df.collect()

for row in object_list:
    table_id = row["table_id"]
    table_name = row["table_name"]
    join_key = row["join_key"]

    if table_name == "orders":
        order_information_func(table_id, table_name, join_key)
        

In [0]:
def order_information_func(table_id, table_name, join_key):

    # Get the last successfull processed timestamp
    last_processed_at = last_successful_processed_ts(table_id)

    # Read the source tables
    silver_orders_current = spark.read.table("silver.transformed.orders_transformed")
    silver_payment_current = spark.read.table("silver.transformed.payments_transformed")
    silver_product_current = spark.read.table("silver.transformed.products_transformed")

    if not last_processed_at:
        changed_orders = silver_orders_current.filter(
            F.col("silver_processed_at") > F.lit(last_processed_at)
        )
        changed_payment = silver_payment_current.filter(
            F.col("silver_processed_at") > F.lit(last_processed_at)
        )
        changed_product = silver_product_current.filter(
            F.col("silver_processed_at") > F.lit(last_processed_at)
        )
    else:
        changed_orders = silver_orders_current
        changed_payment = silver_payment_current
        changed_product = silver_product_current

    print(
        f"Changed Orders: {changed_orders.count()}, Changed Payment: {changed_payment.count()}, Changed Product: {changed_product.count()}"
    )

    impacted_orders = changed_orders.select("order_id").distinct()
    impacted_payments = changed_payment.select("order_id").distinct()
    impacted_products = (
        changed_product.alias("p")
        .join(changed_orders.alias("o"), on="product_id", how="inner")
        .select("p.order_id")
        .distinct()
    )

    impacted_order_ids = (
        impacted_orders.union(impacted_payments).union(impacted_products).distinct()
    )

    silver_impacted_orders = silver_orders_current.join(
        impacted_order_ids, on="order_id", how="inner"
    )

    gold_order_df = (
        silver_impacted_orders.alias("o")
        .join(silver_product_current.alias("p"), "product_id", "inner")
        .join(silver_payment_current.alias("py"), "order_id", "left")
        .select(
            F.col("o.order_id"),
            F.to_date(F.col("o.created_at")).alias("order_date"),
            F.col("o.customer_id"),
            F.col("o.product_id"),
            F.col("p.product_name"),
            F.col("o.order_status"),
            F.col("o.order_amount"),
            F.col("py.payment_id"),
            F.col("py.payment_status"),
            F.col("py.paid_amount"),
            F.lit(gold_run_id).alias("gold_run_id"),
            F.current_timestamp().alias("gold_updated_at"),
        )
        .dropDuplicates(["order_id"])
        .withColumn(
            "payment_ratio",
            F.col("paid_amount") / F.nullif(F.col("order_amount"), 0),
        )
        .withColumn(
            "payment_state",
            F.when(
                F.col("payment_ratio").isNull()
                | (F.col("order_amount") == F.lit(0))
                | (F.col("payment_ratio") < F.lit(0)),
                F.lit("Invalid Payment"),
            ),
        )
        .when(F.col("payment_ratio") == F.lit(0), F.lit("Unpaid"))
        .when(F.col("payment_ratio") == F.lit(1), F.lit("Paid"))
        .when(F.col("payment_ratio") > F.lit(1) & F.col("Overpaid"))
        .when(F.col("payment_ratio") < F.lit(1), F.lit("Underpaid"))
        .otherwise(
            F.lit("Unknown"),
        )
    )

    # Write the data to the gold table
    if not gold_order_df.isEmpty():
        upsert_to_gold(gold_order_df, "gold.curated.order_information", join_key)
    else:
        print("No records to insert")

    # SCD Type 2 implementation
    if not gold_order_df.isEmpty():
        gold_order_df.createOrReplaceTempView("gold_order_vw")

        if not spark.catalog.tableExists("gold.curated.order_information_scd2"):
            spark.sql(
                """
                    CREATE TABLE gold.curated.order_information_scd2 AS
                    SELECT *, 
                        cast(NULL as timestamp) AS valid_from,
                        cast(NULL as timestamp) as valid_to,
                        cast(NULL as string) as current_flag
                    FROM gold.curated.order_information
                    WHERE 1 = 0
                    """
            )

        spark.sql(
            """
                MERGE INTO gold.curated.order_information_scd2 t
                    USING gold_order_vw s ON t.order_id = s.order_id AND t.current_flag = 'Y'
                WHEN MATCHED AND
                (
                    NOT(s.order_date <=> t.order_date)
                    OR NOT(s.customer_id <=> t.customer_id)
                    OR NOT(s.product_id <=> t.product_id)
                    OR NOT(s.product_name <=> t.product_name)
                    OR NOT(s.order_status <=> t.order_status)
                    OR NOT(s.order_amount <=> t.order_amount)
                    OR NOT(s.payment_id <=> t.payment_id)
                    OR NOT(s.payment_status <=> t.payment_status)
                    OR NOT(s.paid_amount <=> t.paid_amount)
                )
                THEN
                    UPDATE SET 
                    valid_to = now(),
                    current_flag = 'N';
                
                INSERT INTO gold.curated.order_information_scd2
                    SELECT *, 
                        now() AS valid_from,
                        cast(NULL as timestamp) as valid_to,
                        'Y' AS current_flag
                    FROM gold_order_vw s
                    LEFT JOIN gold.curated.order_information_scd2 t ON s.order_id = t.order_id
                    AND t.current_flag = 'Y'
                    WHERE (t.order_id IS NULL) OR
                    (
                        NOT(s.order_date <=> t.order_date)
                        OR NOT(s.customer_id <=> t.customer_id)
                        OR NOT(s.product_id <=> t.product_id)
                        OR NOT(s.product_name <=> t.product_name)
                        OR NOT(s.order_status <=> t.order_status)
                        OR NOT(s.order_amount <=> t.order_amount)
                        OR NOT(s.payment_id <=> t.payment_id)
                        OR NOT(s.payment_status <=> t.payment_status)
                        OR NOT(s.paid_amount <=> t.paid_amount) 
                    )
                    """
        )

        ## Load data to Volumes
        latest_snapshot = (
            "/Volumes/gold/curated/snapshots/lates_snapshot/order_information"
        )

        historical_snapshot = f"/Volumes/gold/curated/snapshots/historical_snapshot/order_information/date={datetime.utcnow().strftime('%Y-%m-%d')}/time = {datetime.utcnow().strftime('%H-%M-%S')}"

        spark.read.table("gold.curated.order_information").write.format("parquet").save(
            latest_snapshot
        )
        spark.read.table("gold.curated.order_information_scd2").write.format(
            "parquet"
        ).save(historical_snapshot)

        ## Upsert gold_processing_table

        max_ts = (
            changed_orders.agg(F.max("silver_ingested_at").alias("max_ts"))
            .orderBy(F.col("max_ts").desc())
            .limit(1)
        ).collect()[0]["max_ts"]

        upsert_gold_control(table_id, max_ts, gold_order_df.count(), "success")